# LLM 모델 선택 평가 - fine-tuning 필요 여부 검증

**작성일**: 2026-08-31
**목적**: 팀이 계획 중인 fine-tuning(후보 3모델을 fine-tuning 후 RunPod Serverless로
서빙) 전에, **fine-tuning 없이 프롬프팅만으로 되는지**를 먼저 확인한다.

## 왜 이게 필요한가

이 프로젝트의 그래프 파이프라인(N1~N14) 중 4개 노드가 LLM을 쓴다.

| 노드 | 역할 | LLM이 하는 일 | LLM 없이도 동작하나? |
|---|---|---|---|
| N5 `claim_plan` | 정책 원문에서 claim 후보 추출 | 원문에서 자격/금액/중복수급 근거 문장을 JSON으로 추출 | 예 (원문 전체를 그대로 씀, 정밀도만 낮음) |
| N9 `eligibility_verdict` | 위반 사유 자연어화 | 규칙이 만든 기계적 문장을 자연스러운 한국어로 다듬음 | 예 (기계적 문장 그대로 노출) |
| N10 `benefit_calculator` | 지원금액 추출 | 비정형 텍스트에서 금액을 JSON으로 추출 | 예 (금액 null 처리) |
| N13 `answer_generation` | 최종 답변 다듬기 | 템플릿 문장을 자연스럽게 다듬음 | 예 (템플릿 문장 그대로 노출) |

**중요**: 네 노드 모두 "판정/사실 자체"는 규칙 기반으로 이미 확정돼 있고, LLM은
표현만 다듬거나 이미 있는 텍스트에서 정보를 추출하는 보조 역할이다. LLM 호출이
실패하면 전부 규칙 기반/템플릿으로 안전하게 폴백하도록 이미 설계돼 있다 - 그래서
"이 LLM이 이 프롬프트에 안 되면 서비스가 죽는다"가 아니라, "이 LLM이 되면 답변
품질이 더 좋아진다"는 관계다.

## 이 노트북이 하는 일

1. **1부**: 2026-08-31에 이미 수행한 3회 실행 기록(`scripts/llm_prompt_probe.py`)을
   그대로 임베딩해서 평가지표를 계산한다 (API 재호출 없음, 무료로 바로 확인 가능).
2. **2부**: 원하면 지금 이 컴퓨터에서 직접 API를 다시 호출해서 데이터를 더 쌓을 수
   있다 (HuggingFace 계정과 크레딧 필요 - 아래 "주의" 참고).
3. **3부**: 두 데이터를 합쳐서 종합 지표/그래프를 다시 계산하고, 모델 선택 기준과
   현재까지의 결론을 정리한다.

## ⚠️ 실행 전 주의사항

- **HuggingFace 무료 tier 크레딧은 월 $0.10 밖에 안 된다** (`Qwen/Qwen3.5-9B`처럼
  "추론(thinking)형" 모델은 답을 쓰기 전에 내부 사고 과정에 토큰을 많이 써서
  이 크레딧을 순식간에 다 쓴다 - 2026-08-31 3차 실행에서 실제로 다 써서
  `402 Payment Required`를 받았다). 2부(재실행) 셀은 기본적으로 **꺼져 있다**
  (`RUN_LIVE = False`). 켜기 전에 `https://huggingface.co/settings/billing`에서
  남은 크레딧을 확인하는 걸 권장한다.
- `Bllossom/llama-3.2-Korean-Bllossom-3B`, `skt/A.X-4.0-Light`는
  `https://huggingface.co/settings/inference-providers`에서 해당 모델을 서빙하는
  provider를 계정에서 켜야 호출이 된다 (안 켜면 `model_not_supported` 에러).
  2026-08-31 기준 이 두 모델은 **한 번도 성공적으로 호출된 적이 없다** - 아래
  1부의 결과가 전부 "차단됨"인 이유다.
- `.env` 파일에 `HF_TOKEN`을 넣어두면 자동으로 읽힌다 (`.env.example` 참고).
  토큰에는 "Make calls to Inference Providers" 권한이 따로 켜져 있어야 한다.


## 0. 환경 설정

레포 루트(`bokji-agent/`)에서 이 노트북을 실행해야 한다 (상대 import 때문에).
아래 셀은 필요한 패키지를 설치하고, 이 노드들이 실제로 쓰는 프로덕션 함수를
그대로 import한다 - 이 노트북이 별도로 프롬프트를 만드는 게 아니라, 실제
그래프에 붙었을 때 나가는 프롬프트 그대로를 평가한다.

In [ ]:
%pip install -q pandas matplotlib huggingface_hub python-dotenv


In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# 레포 루트를 sys.path에 추가 (scripts/llm_prompt_probe.py와 동일한 관례)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    # 노트북을 notebooks/ 안에서 열었을 경우 상위 폴더로 보정
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from rag_design.contracts import (
    Chunk,
    RetrievedChunk,
    SCHEMA_VERSION,
    SourceType,
    compute_content_hash,
)
from src.rag_chatbot.graph.nodes.answer_generation import generate_answer
from src.rag_chatbot.graph.nodes.benefit_calculator import _extract_amount_via_llm
from src.rag_chatbot.graph.nodes.claim_extractor import (
    LLMClaimExtractor,
    RuleBasedClaimExtractor,
)
from src.rag_chatbot.graph.nodes.eligibility_verdict import _naturalize_reasons
from src.rag_chatbot.llm import HuggingFaceInferenceClient, LLMCallError

CANDIDATE_MODELS = [
    "Bllossom/llama-3.2-Korean-Bllossom-3B",
    "skt/A.X-4.0-Light",
    "Qwen/Qwen3.5-9B",
]

SAMPLE_POLICY_TEXT = (
    "지원대상\n"
    "만 65세 이상 저소득 어르신 중 소득인정액이 기준 중위소득 50% 이하인 자.\n"
    "지원내용\n"
    "1인당 월 200,000원을 매월 20일 지급한다. 타 유사 현금성 지원과 "
    "중복수급은 불가하다."
)
SAMPLE_RULE_REASONS = [
    "연령 조건 미충족: 최소 65세부터 지원 (사용자 age=40)",
    "소득 조건 미충족: 기준 중위소득 50% 이하만 지원 (사용자 income_bracket=pct_100_150)",
]
EXPECTED_AMOUNT = 200000.0

# 그래프에 한글 라벨이 있어서, 시스템에 한글 폰트가 있으면 그걸 쓰도록 시도한다
# (없으면 그냥 기본 폰트로 - 글자가 네모로 깨져 보일 뿐 에러는 안 남).
import matplotlib.font_manager as fm

_KOREAN_FONT_CANDIDATES = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
_available = {f.name for f in fm.fontManager.ttflist}
for _name in _KOREAN_FONT_CANDIDATES:
    if _name in _available:
        plt.rcParams["font.family"] = _name
        break
plt.rcParams["axes.unicode_minus"] = False

print("설정 완료. REPO_ROOT =", REPO_ROOT)


## 1. 평가지표 정의

각 (모델, 노드) 호출마다 아래 항목을 기록한다.

| 지표 | 의미 |
|---|---|
| `outcome` | `success`(LLM 응답을 실제로 씀) / `fallback`(규칙 기반·템플릿으로 대체됨) |
| `error_type` | 실패 원인 분류 - 아래 참고 |
| `facts_preserved` | 원문의 숫자/조건(나이 65/40, 소득구간 100~150%, 금액 200,000원 등)이 그대로 보존됐는지 (`True`/`False`/`None`=해당없음) |
| `latency_sec` | 호출 1회 소요 시간(초). 과거 기록은 미측정(`None`) - 재실행부터 자동 측정 |

`error_type` 분류 (모델의 "이해력" 문제가 아니라 **환경/설정 문제**인 것과,
모델의 실제 프롬프트 수행 능력과 관련된 것을 구분한다):

- `provider_not_enabled` — 환경 문제. 계정에서 이 모델의 provider를 안 켜서
  요청 자체가 거부됨(`model_not_supported`). 모델 능력과 무관.
- `billing_exhausted` — 환경 문제. 월간 무료 크레딧 소진(`402`). 모델 능력과 무관.
- `timeout` — 환경 문제일 수도, 모델이 너무 오래 추론해서일 수도 있음(모호).
- `token_budget_exceeded` — **모델 특성**. "추론형" 모델이 답을 쓰기 전에
  사고 과정에 토큰을 다 써서 최종 답이 비어서 나옴(`finish_reason='length'`).
  같은 프롬프트도 실행마다 갈릴 수 있음(비결정적).
- `validation_failed` — **모델 능력 관련**. LLM은 응답했지만 JSON 파싱이나
  "원문 그대로 인용" 검증을 통과 못 해서 폴백됨.
- `None` (성공) — 정상적으로 LLM 응답을 그대로 씀.

**중요한 구분**: `provider_not_enabled`/`billing_exhausted`는 이 모델이 "이 작업을
할 수 있는지"에 대해 아무것도 말해주지 않는다 - 단지 아직 시도조차 못 해봤다는
뜻이다. 반면 `token_budget_exceeded`/`validation_failed`는 실제로 모델이 이
프롬프트를 다뤄본 결과이므로 평가에 의미가 있다.

## 2. 1부 - 과거 실행 기록 (2026-08-31, 3회)

`scripts/llm_prompt_probe.py`를 이미 3번 실행한 실제 결과를 그대로 옮겨왔다
(지어낸 데이터 아님 - 실행 로그에서 그대로 옮김). 각 실행은 `max_new_tokens`
설정이 달랐다 (모델의 응답이 잘리는 문제를 진단하며 4096 → 8192로 올렸다).

- **1차 실행**: `max_new_tokens=4096`, 진단 로그 없음(왜 폴백됐는지 원인 미상)
- **2차 실행**: `max_new_tokens=4096`, 진단 래퍼 추가 후(원인까지 기록됨)
- **3차 실행**: `max_new_tokens=8192`, `Qwen3.5-9B`가 크레딧 소진으로 중단됨

In [ ]:
HISTORICAL_RUNS = [
    # ---------------- 1차 실행 (max_new_tokens=4096, 진단 로그 없음) ----------------
    dict(run="1차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N5",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N9",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N10",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N13",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="skt/A.X-4.0-Light", node="N5",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="skt/A.X-4.0-Light", node="N9",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="skt/A.X-4.0-Light", node="N10",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="skt/A.X-4.0-Light", node="N13",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="1차", model="Qwen/Qwen3.5-9B", node="N5",
         outcome="fallback", error_type="unknown(진단 로그 도입 전)", facts_preserved=None),
    dict(run="1차", model="Qwen/Qwen3.5-9B", node="N9",
         outcome="success", error_type=None, facts_preserved=True,
         note="65세/40세, 중위소득 100~150% 그대로 보존하며 자연스럽게 문장화"),
    dict(run="1차", model="Qwen/Qwen3.5-9B", node="N10",
         outcome="success", error_type=None, facts_preserved=True,
         note="amount=200000.0 정확히 추출"),
    dict(run="1차", model="Qwen/Qwen3.5-9B", node="N13",
         outcome="fallback", error_type="unknown(진단 로그 도입 전, 다른 실패 사례와 동일 템플릿 출력이라 폴백으로 판단)",
         facts_preserved=None),

    # ---------------- 2차 실행 (max_new_tokens=4096, 진단 래퍼 도입) ----------------
    dict(run="2차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N5",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N9",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N10",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N13",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="skt/A.X-4.0-Light", node="N5",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="skt/A.X-4.0-Light", node="N9",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="skt/A.X-4.0-Light", node="N10",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="skt/A.X-4.0-Light", node="N13",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="2차", model="Qwen/Qwen3.5-9B", node="N5",
         outcome="fallback", error_type="token_budget_exceeded", facts_preserved=None,
         note="finish_reason='length', max_new_tokens=4096으로도 부족 (claim_type 3개 한 번에 요구하는 가장 무거운 프롬프트)"),
    dict(run="2차", model="Qwen/Qwen3.5-9B", node="N9",
         outcome="fallback", error_type="token_budget_exceeded", facts_preserved=None,
         note="1차에서는 성공했던 동일 프롬프트가 이번엔 finish_reason='length'로 실패 - 실행마다 결과가 달라짐(비결정적)"),
    dict(run="2차", model="Qwen/Qwen3.5-9B", node="N10",
         outcome="success", error_type=None, facts_preserved=True,
         note="amount=200000.0 정확히 추출 (1차와 동일하게 안정적)"),
    dict(run="2차", model="Qwen/Qwen3.5-9B", node="N13",
         outcome="success", error_type=None, facts_preserved=True,
         note="'지원 자격이 충족되어 지원 가능합니다. 지원 금액은 200,000원이며...' - 사실 안 바꾸고 자연스럽게 패러프레이즈"),

    # ---------------- 3차 실행 (max_new_tokens=8192, Qwen 크레딧 소진으로 중단) ----------------
    dict(run="3차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N5",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N9",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N10",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="Bllossom/llama-3.2-Korean-Bllossom-3B", node="N13",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="skt/A.X-4.0-Light", node="N5",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="skt/A.X-4.0-Light", node="N9",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="skt/A.X-4.0-Light", node="N10",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="skt/A.X-4.0-Light", node="N13",
         outcome="fallback", error_type="provider_not_enabled", facts_preserved=None),
    dict(run="3차", model="Qwen/Qwen3.5-9B", node="N5",
         outcome="fallback", error_type="timeout", facts_preserved=None,
         note="클라이언트 timeout(60초) 초과 - 8192 토큰까지 추론이 길어진 것으로 추정"),
    dict(run="3차", model="Qwen/Qwen3.5-9B", node="N9",
         outcome="fallback", error_type="billing_exhausted", facts_preserved=None,
         note="402 Payment Required - 월간 무료 크레딧($0.10) 소진, 이 시점부터 이 모델은 더 이상 테스트 불가"),
    dict(run="3차", model="Qwen/Qwen3.5-9B", node="N10",
         outcome="fallback", error_type="billing_exhausted", facts_preserved=None),
    dict(run="3차", model="Qwen/Qwen3.5-9B", node="N13",
         outcome="fallback", error_type="billing_exhausted", facts_preserved=None),
]

for r in HISTORICAL_RUNS:
    r.setdefault("latency_sec", None)
    r.setdefault("note", "")

hist_df = pd.DataFrame(HISTORICAL_RUNS)
hist_df["source"] = "historical"
print(f"과거 기록 {len(hist_df)}건 로드 완료 (3회 실행 x 3모델 x 4노드)")
hist_df.head()


## 3. 1부 결과 요약

아래는 위 과거 기록만으로 계산한 지표다 (API 호출 없이 바로 확인 가능).

In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """(모델, 노드)별 성공률과 지배적 오류 유형을 계산한다."""
    rows = []
    for (model, node), group in df.groupby(["model", "node"]):
        n = len(group)
        n_success = (group["outcome"] == "success").sum()
        error_counts = group.loc[group["outcome"] == "fallback", "error_type"].value_counts()
        dominant_error = error_counts.index[0] if len(error_counts) else None
        # provider_not_enabled/billing_exhausted는 "모델이 이 작업을 못 한다"는
        # 뜻이 아니라 "아직 못 시도했다"는 뜻이므로, 환경 요인만으로 실패한
        # 시도는 능력 평가용 성공률 분모에서 뺀다.
        env_only = group["error_type"].isin(["provider_not_enabled", "billing_exhausted"])
        n_evaluable = n - env_only.sum()
        capability_rate = (n_success / n_evaluable) if n_evaluable else None
        rows.append(
            dict(
                model=model,
                node=node,
                시도=n,
                성공=n_success,
                전체_성공률=round(n_success / n, 2) if n else None,
                평가가능_시도=n_evaluable,
                평가가능_성공률=round(capability_rate, 2) if capability_rate is not None else None,
                지배적_오류=dominant_error,
            )
        )
    return pd.DataFrame(rows)


summary_df = summarize(hist_df)
summary_df


In [ ]:
# 모델별 provider 상태(= 아직 평가 자체가 됐는지) 한눈에 보기
readiness = (
    hist_df.groupby("model")["error_type"]
    .apply(lambda s: "provider_not_enabled" not in set(s))
    .rename("이 모델이 최소 한 번이라도 실제로 호출됨")
    .to_frame()
)
readiness


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pivot = hist_df.assign(is_success=(hist_df["outcome"] == "success").astype(int)).pivot_table(
    index="node", columns="model", values="is_success", aggfunc="sum"
)
pivot = pivot.reindex(["N5", "N9", "N10", "N13"])
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("성공 횟수 (총 3회 시도 중)")
ax.set_title("노드별 x 모델별 성공 횟수 (1부: 과거 3회 실행 기준)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 1부 해석

- **`Bllossom/llama-3.2-Korean-Bllossom-3B`, `skt/A.X-4.0-Light`**: 3회 실행 x
  4노드 = 12번 시도 전부 `provider_not_enabled`. **이 두 모델은 단 한 번도
  실제로 평가되지 못했다** - "프롬프팅으로 안 된다"가 아니라 "아직 시도조차
  못 했다"가 정확한 표현이다. `https://huggingface.co/settings/inference-providers`
  에서 provider를 켜야 평가가 시작된다.
- **`Qwen/Qwen3.5-9B`**:
  - N10(금액 추출)은 크레딧이 남아있던 2번 모두 성공, 사실도 정확 - **가장
    신뢰할 만한 신호**.
  - N9(자연어화)는 1차 성공 → 2차 실패(같은 프롬프트인데 결과가 갈림) →
    3차는 크레딧 소진으로 시도 자체가 안 됨. **모델 능력은 있어 보이지만
    호출마다 안정적이지 않다.**
  - N13(답변 다듬기)도 N9와 비슷한 패턴 - 1차 실패, 2차 성공(사실 보존 확인),
    3차 크레딧 소진.
  - N5(claim 3종류 동시 추출)는 **3번 모두 실패** - 이유는 매번 달랐지만
    (진단 로그 도입 전 불명 / 토큰 예산 초과 / timeout), 결과적으로 네 노드
    중 유일하게 단 한 번도 성공하지 못한 노드다. 프롬프트가 요구하는 작업
    자체가 가장 무겁다(claim_type 3개 + 원문 그대로 인용 + JSON 스키마
    준수를 한 번에 요구).
- Qwen3.5-9B가 "추론(thinking)형" 모델이라는 점이 핵심 리스크다 - 최종 답을
  쓰기 전에 내부 사고 과정에 토큰을 쓰는데, 이 길이가 매 호출 다르다
  (비결정적). 그래서 같은 프롬프트가 어떤 날은 성공하고 어떤 날은 토큰
  예산을 넘겨 실패하며, 이게 무료 크레딧을 예측 불가능하게 빨리 소진시킨다
  (3차 실행에서 실제로 발생). 이건 "모델이 작업을 이해하는가"와는 별개의
  **운영 안정성/비용 문제**다.

## 4. 2부 - 지금 직접 재실행하기 (선택, API 비용 발생)

아래 셀은 기본적으로 꺼져 있다(`RUN_LIVE = False`). 직접 API를 다시 호출해서
데이터를 더 쌓고 싶으면:

1. 레포 루트 `.env`에 본인의 `HF_TOKEN`을 넣는다 (토큰에 "Make calls to
   Inference Providers" 권한 필요).
2. `Bllossom`/`A.X-4.0-Light`를 테스트하려면 먼저
   `https://huggingface.co/settings/inference-providers`에서 provider를 켠다.
3. `https://huggingface.co/settings/billing`에서 남은 크레딧을 확인한다
   (무료 tier는 월 $0.10 - Qwen3.5-9B처럼 추론형 모델은 몇 번만 호출해도
   소진될 수 있다).
4. 아래 셀에서 `RUN_LIVE = True`로 바꾸고, 필요하면 `REPEAT_COUNT`(모델x노드당
   반복 횟수)를 조절한 뒤 실행한다. `REPEAT_COUNT`를 늘릴수록 Qwen처럼
   비결정적인 모델의 실제 성공률을 더 정확히 알 수 있지만 크레딧도 그만큼
   더 쓴다.

In [ ]:
RUN_LIVE = False  # True로 바꾸면 실제 HuggingFace Inference API를 호출한다 (크레딧 소모).
REPEAT_COUNT = 1  # (모델, 노드)당 반복 시도 횟수. 늘릴수록 신뢰도는 올라가지만 비용도 늘어난다.


In [ ]:
def _sample_chunk(chunk_id: str, source_url: str) -> Chunk:
    return Chunk(
        schema_version=SCHEMA_VERSION,
        chunk_id=chunk_id,
        doc_id="probe-policy",
        source_type=SourceType.SUBSIDY,
        text=SAMPLE_POLICY_TEXT,
        heading_path=("지원대상",),
        ordinal=0,
        citation_locator="지원대상",
        content_hash=compute_content_hash(SAMPLE_POLICY_TEXT),
        metadata={"source_url": source_url},
    )


def _sample_state_for_n13() -> dict:
    chunk = _sample_chunk("probe-policy-chunk-1", "https://www.gov.kr/portal/rcvfvrSvc/dtlEx/probe")
    retrieved = RetrievedChunk(
        query_id="probe", chunk=chunk, rank=1, score=0.1,
        score_type="cosine_distance", retriever_version="probe:fixture", index_name="subsidy",
    )
    return {
        "assembled_result": {
            "policies": {
                "probe-policy": {
                    "eligibility": {"policy_id": "probe-policy", "verdict": "충족", "reasons": ["근거 문장"]},
                    "benefit_amount": {"policy_id": "probe-policy", "amount": 200000.0},
                    "duplicate": {"policy_id": "probe-policy", "status": "미확인"},
                }
            }
        },
        "claim_plan": [
            {"claim_id": "c1", "policy_id": "probe-policy", "claim_type": "eligibility",
             "evidence_chunk_ids": ["probe-policy-chunk-1"]}
        ],
        "subsidy_chunks": [retrieved],
        "law_chunks": [],
        "node_trace": [],
    }


def _facts_preserved_n9(naturalized: list[str]) -> bool:
    """65/40세, 중위소득 100~150% 같은 핵심 숫자가 살아있는지 대략 확인한다
    (완벽한 검증은 아니다 - 최종 판단은 눈으로 결과를 읽고 하는 걸 권장)."""
    joined = " ".join(naturalized)
    return all(token in joined for token in ["65", "40"])


def _facts_preserved_n13(draft_answer: str) -> bool:
    return "200000" in draft_answer.replace(",", "") or "200,000" in draft_answer


def probe_once(model_name: str) -> list[dict]:
    """모델 1개에 대해 N5/N9/N10/N13을 한 번씩 호출하고 기록 리스트를 반환한다."""
    records: list[dict] = []
    try:
        client = HuggingFaceInferenceClient(model=model_name)
    except ValueError as exc:
        for node in ["N5", "N9", "N10", "N13"]:
            records.append(dict(model=model_name, node=node, outcome="fallback",
                                 error_type="client_init_failed", facts_preserved=None,
                                 latency_sec=None, note=str(exc)))
        return records

    # N5
    t0 = time.perf_counter()
    try:
        claims = LLMClaimExtractor(client).extract(policy_id="probe-policy", text=SAMPLE_POLICY_TEXT)
        fell_back = claims == RuleBasedClaimExtractor().extract(policy_id="probe-policy", text=SAMPLE_POLICY_TEXT)
        records.append(dict(model=model_name, node="N5", outcome=("fallback" if fell_back else "success"),
                             error_type=("validation_or_call_failed" if fell_back else None),
                             facts_preserved=(None if fell_back else True),
                             latency_sec=round(time.perf_counter() - t0, 2), note=str(claims)[:200]))
    except Exception as exc:  # noqa: BLE001
        records.append(dict(model=model_name, node="N5", outcome="fallback", error_type=type(exc).__name__,
                             facts_preserved=None, latency_sec=round(time.perf_counter() - t0, 2), note=str(exc)[:200]))

    # N9
    t0 = time.perf_counter()
    try:
        naturalized = _naturalize_reasons(SAMPLE_RULE_REASONS, client)
        fell_back = naturalized == SAMPLE_RULE_REASONS
        records.append(dict(model=model_name, node="N9", outcome=("fallback" if fell_back else "success"),
                             error_type=("validation_or_call_failed" if fell_back else None),
                             facts_preserved=(None if fell_back else _facts_preserved_n9(naturalized)),
                             latency_sec=round(time.perf_counter() - t0, 2), note=str(naturalized)[:200]))
    except Exception as exc:  # noqa: BLE001
        records.append(dict(model=model_name, node="N9", outcome="fallback", error_type=type(exc).__name__,
                             facts_preserved=None, latency_sec=round(time.perf_counter() - t0, 2), note=str(exc)[:200]))

    # N10
    t0 = time.perf_counter()
    try:
        amount, note = _extract_amount_via_llm(SAMPLE_POLICY_TEXT, client)
        success = amount == EXPECTED_AMOUNT
        records.append(dict(model=model_name, node="N10", outcome=("success" if success else "fallback"),
                             error_type=(None if success else "wrong_or_missing_amount"),
                             facts_preserved=success,
                             latency_sec=round(time.perf_counter() - t0, 2), note=note[:200]))
    except Exception as exc:  # noqa: BLE001
        records.append(dict(model=model_name, node="N10", outcome="fallback", error_type=type(exc).__name__,
                             facts_preserved=None, latency_sec=round(time.perf_counter() - t0, 2), note=str(exc)[:200]))

    # N13
    t0 = time.perf_counter()
    try:
        result = generate_answer(_sample_state_for_n13(), llm_client=client)
        draft = result.get("draft_answer", "")
        template_marker = "[probe-policy]\n- 지원자격: 지원 가능"
        fell_back = draft.startswith(template_marker)
        records.append(dict(model=model_name, node="N13", outcome=("fallback" if fell_back else "success"),
                             error_type=("llm_failed_or_empty" if fell_back else None),
                             facts_preserved=(None if fell_back else _facts_preserved_n13(draft)),
                             latency_sec=round(time.perf_counter() - t0, 2), note=draft[:200]))
    except Exception as exc:  # noqa: BLE001
        records.append(dict(model=model_name, node="N13", outcome="fallback", error_type=type(exc).__name__,
                             facts_preserved=None, latency_sec=round(time.perf_counter() - t0, 2), note=str(exc)[:200]))

    return records


live_records: list[dict] = []
if RUN_LIVE:
    for model_name in CANDIDATE_MODELS:
        for rep in range(1, REPEAT_COUNT + 1):
            print(f"[실행 중] {model_name} - {rep}/{REPEAT_COUNT}회차")
            for rec in probe_once(model_name):
                rec["run"] = f"라이브-{rep}"
                live_records.append(rec)
    live_df = pd.DataFrame(live_records)
    live_df["source"] = "live"
    print(f"라이브 실행 {len(live_df)}건 완료")
else:
    live_df = pd.DataFrame(columns=hist_df.columns)
    print("RUN_LIVE=False - 2부를 건너뜁니다 (1부 과거 기록만으로 아래 3부를 계산합니다).")


## 5. 3부 - 종합 지표 & 모델 선택 사유

과거 기록(1부) + 방금 실행한 라이브 기록(2부, 실행했다면)을 합쳐서 다시 계산한다.

In [ ]:
all_df = pd.concat([hist_df, live_df], ignore_index=True)
final_summary = summarize(all_df)
final_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pivot = all_df.assign(is_success=(all_df["outcome"] == "success").astype(int)).pivot_table(
    index="node", columns="model", values="is_success", aggfunc="mean"
)
pivot = pivot.reindex(["N5", "N9", "N10", "N13"])
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("성공률 (전체 시도 기준)")
ax.set_ylim(0, 1)
ax.set_title("노드별 x 모델별 성공률 (과거+라이브 전체)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 모델 선택 판단 기준 (제안)

아래 기준으로 "fine-tuning 없이 프롬프팅만으로 충분한가"를 판단할 것을 제안한다.
팀에서 임계값은 조정 가능하다.

1. **최소 조건**: 후보 모델이 실제로 호출 가능해야 한다(`provider_not_enabled`가
   아니어야 함) - 이게 안 되면 아예 평가 대상이 아니다.
2. **성공률**: `평가가능_성공률`(환경 요인 제외한 성공률)이 노드별로 최소
   80% 이상이어야 프롬프팅만으로 실사용 가능하다고 볼 수 있다. 그 이하면
   few-shot 예시 추가, 프롬프트 재설계, 또는 fine-tuning을 고려해야 한다.
3. **사실 보존**: `facts_preserved`가 성공 케이스에서 100%여야 한다(하나라도
   숫자/조건이 바뀌면 그 모델은 이 프롬프트로는 못 믿는다 - 복지 정보
   안내라는 도메인 특성상 이 기준은 타협 불가).
4. **안정성**: 같은 프롬프트를 여러 번 돌렸을 때 결과가 일관돼야 한다
   (Qwen3.5-9B처럼 실행마다 성공/실패가 갈리면, 성공률이 평균적으로 높아도
   프로덕션에는 부적합할 수 있다 - 재시도 로직이나 더 큰 토큰 예산으로
   완화 가능한지 별도 검토 필요).

### 현재 데이터 기준 잠정 결론 (2026-08-31)

- **아직 최종 결론을 내릴 수 없다.** 후보 3모델 중 2개(`Bllossom`,
  `A.X-4.0-Light`)가 단 한 번도 평가되지 못했다 - provider 활성화가 먼저
  필요하다.
- `Qwen/Qwen3.5-9B`만 보면: N10은 기준을 사실상 만족(성공률 100%, 사실 보존
  100%). N9/N13은 성공했을 때 품질은 좋지만 표본이 너무 적어(2~3회) 위 기준
  (80% 이상, 안정성)을 만족하는지 판단하기엔 이르다. N5는 현재까지 100%
  실패 - 이 노드는 프롬프트나 실행 전략(예: claim_type별로 나눠서 3번 호출)
  자체를 재검토할 필요가 있어 보인다.
- Qwen3.5-9B가 "추론형" 모델이라는 특성상, 토큰 예산을 계속 늘리는 방향보다는
  **비추론형 instruct 모델(Bllossom, A.X-4.0-Light)이 오히려 이 4개 노드의
  단순 추출/다듬기 작업에는 더 안정적이고 저렴할 가능성**이 있다 - 다만
  아직 이 두 모델의 데이터가 전혀 없어서 확인이 안 된 가설이다.

### 다음 액션

1. `Bllossom`, `A.X-4.0-Light`의 provider를 계정에서 켜고, 크레딧을 확보한 뒤
   `RUN_LIVE = True`로 재실행 (`REPEAT_COUNT`를 3~5 정도로 늘려서 표본을
   충분히 확보하는 걸 권장).
2. `Qwen/Qwen3.5-9B`는 N5/N9에서 재현되는 실패가 진짜 "능력 부족"인지
   "토큰 예산/비결정성" 문제인지 더 데이터를 쌓아 구분.
3. 위 기준(성공률 80%, 사실 보존 100%, 안정성)을 만족하는 모델이 하나라도
   나오면 "프롬프팅만으로 충분"으로 잠정 결론, 그렇지 않으면 fine-tuning
   또는 프롬프트/노드 구조 재설계를 검토.

## 부록 - 각 노드가 실제로 보내는 프롬프트 (재현성/투명성)

이 노트북이 만든 프롬프트가 아니라, 프로덕션 코드(`src/rag_chatbot/graph/nodes/`)가
실제로 만드는 프롬프트를 그대로 가져온 것이다. 다른 모델을 테스트하거나 프롬프트
자체를 개선하려는 사람이 참고할 수 있게 남겨둔다.

- **N5** (`claim_extractor.py`의 `LLMClaimExtractor`): 정책 원문 전체를 주고
  eligibility/amount/duplicate 3종류의 claim을 JSON으로 뽑되, `reasons`는
  반드시 원문에서 그대로 발췌해야 한다고 명시. 이 파일은 LLM이 그 제약을
  지켰는지 스스로 검증하지 않고(원문에 실제로 포함된 문장만 통과시키는
  1차 필터만 있음), N6(`document_verification`)이 다시 한번 검증한다.
- **N9** (`eligibility_verdict.py`의 `_naturalize_reasons`): 규칙이 기계적으로
  만든 위반 사유 문장을 "숫자/조건은 절대 바꾸지 말고 표현만" 자연스럽게
  다듬으라고 지시. 응답을 줄 단위로 나눠서 그대로 쓴다 - JSON이 아니라
  일반 텍스트라 N5/N10보다 형식 요구가 느슨하다.
- **N10** (`benefit_calculator.py`의 `_extract_amount_via_llm`): 정책 원문에서
  "이미 명시된" 금액만 JSON(`{"amount": ..., "reason": ...}`)으로 뽑고, 계산이나
  추측은 금지. 조건부 금액이면 `amount: null`로 정직하게 두라고 지시.
- **N13** (`answer_generation.py`의 `generate_answer`): 규칙 기반으로 이미
  검증된 템플릿 문장을 "사실을 하나도 바꾸지 말고" 자연스러운 안내문으로
  다듬으라고 지시. 인용(citation)은 LLM 출력에서 뽑지 않고 별도로 조립한다
  (LLM이 "이 출처를 봤다"고 말해도 근거로 안 씀).
